In [1]:
import os
import zstandard  # pip install zstandard
from tqdm import tqdm
import random
import json
import langid

files_processed_to_text = False




In [2]:
from datetime import datetime
def showTime():
    return str("["+datetime.now().strftime('%Y-%m-%d %H:%M:%S.%f')+" UTC]")

In [3]:
def zst_files_in_dir(directory):
    """List all .zst files in a directory."""
    files = []
    for filename in os.listdir(directory):
        if filename.endswith(".zst") and os.path.isfile(os.path.join(directory, filename)):
            files.append(filename)
    return files

In [4]:
from typing import Generator, Optional, Set

def decompress_zst_to_text(
    input_file: str,
    vocab: Optional[Set[str]] = None,
    mode: str = "accuracy",
    ascii_threshold: float = 0.5
) -> Generator[str, None, None]:
    """
    Decompresses a .zst file containing JSONL (JSON lines) format, 
    and yields English texts filtered via language detection or ASCII checks.
    
    ### Parameters
    input_file (str):
        Path to the .zst file containing JSONL-formatted lines.
        
    vocab (Optional[Set[str]]):
        A vocabulary set to collect unique characters from valid English texts. Defaults to None.
        
    mode (str, default='accuracy'):
        - 'accuracy': Uses the `langid` library for precise English language detection.
        - 'speed': Uses an ASCII ratio check for faster filtering.
        
    ascii_threshold (float, default=0.5):
        Minimum ASCII character ratio (0.0-1.0) for mode='speed' to consider text as valid.
    
    ### Yields
    str:
        Filtered English text entries from the compressed file.
    """
    with open(input_file, "rb") as infile:
        dctx = zstandard.ZstdDecompressor()
        with dctx.stream_reader(infile) as reader:
            current_line = ""
            while True:
                chunk = reader.read(16384).decode("utf-8", errors="replace")  # Read in 16KB chunks
                if not chunk:
                    break
                current_line += chunk
                # Split into lines (handles partial lines)
                lines = current_line.split("\n")
                current_line = lines.pop() if lines else ""  # Save partial line for next iteration

                # Process each line
                for line in lines:
                    # Skip empty lines after stripping
                    stripped_line = line.strip()
                    if not stripped_line:
                        continue
                    
                    try:
                        data = json.loads(stripped_line)
                        text = data.get("text", "").strip()
                    except (json.JSONDecodeError, KeyError):
                        continue  # Skip invalid JSON
                        
                    except Exception as e:
                        print(f"JSON Error: {e} on line: {line[:50]}...")
                        continue

                    if mode == "accuracy":      
                        # Check language (English)
                        try:
                            detected_lang, _ = langid.classify(text)
                        except langid.langid.LanguageIdentificationError:
                            # Skip texts too short to identify
                            continue

                        if detected_lang != "en":
                            continue  # Non-English, skip

                    elif mode == "speed":
                        # ---- START FILTERING LOGIC ----
                        ascii_count = 0
                        total_chars = 0
                        
                        # Iterate through each character in text
                        for c in text:
                            code = ord(c)
                            if code <= 127:
                                ascii_count += 1
                            total_chars += 1

                        # Check filtering conditions
                        if total_chars == 0:
                            continue
                        if (ascii_count / total_chars) < ascii_threshold:
                            continue
                        # ---- END FILTERING LOGIC ----

                    # Update the vocabulary (only for English texts)
                    if vocab is not None:
                        vocab.update(set(text))

                    yield text.strip()


In [5]:
folder_path = "openwebtext2"
output_file = "output_v7_accuracy.txt"
vocab_file = "vocab_v7_accuracy.txt"


In [6]:
# Gather files
files = zst_files_in_dir(folder_path)
total_files = len(files)
print(f"Total files: {total_files}")
print(files)
vocab = set()

Total files: 179
['2005-06.jsonl.zst', '2005-07.jsonl.zst', '2005-08.jsonl.zst', '2005-09.jsonl.zst', '2005-10.jsonl.zst', '2005-11.jsonl.zst', '2005-12.jsonl.zst', '2006-01.jsonl.zst', '2006-02.jsonl.zst', '2006-03.jsonl.zst', '2006-04.jsonl.zst', '2006-05.jsonl.zst', '2006-06.jsonl.zst', '2006-07.jsonl.zst', '2006-08.jsonl.zst', '2006-09.jsonl.zst', '2006-10.jsonl.zst', '2006-11.jsonl.zst', '2006-12.jsonl.zst', '2007-01.jsonl.zst', '2007-02.jsonl.zst', '2007-03.jsonl.zst', '2007-04.jsonl.zst', '2007-05.jsonl.zst', '2007-06.jsonl.zst', '2007-07.jsonl.zst', '2007-08.jsonl.zst', '2007-09.jsonl.zst', '2007-10.jsonl.zst', '2007-11.jsonl.zst', '2007-12.jsonl.zst', '2008-01.jsonl.zst', '2008-02.jsonl.zst', '2008-03.jsonl.zst', '2008-04.jsonl.zst', '2008-05.jsonl.zst', '2008-06.jsonl.zst', '2008-07.jsonl.zst', '2008-08.jsonl.zst', '2008-09.jsonl.zst', '2008-10.jsonl.zst', '2008-11.jsonl.zst', '2008-12.jsonl.zst', '2009-01.jsonl.zst', '2009-02.jsonl.zst', '2009-03.jsonl.zst', '2009-04.jsonl.z

In [7]:
# Shuffle files randomly 
random.seed(42)  # Optional: Set seed for reproducibility
random.shuffle(files)  # Shuffle in-place
print(files)

['2018-02.jsonl.zst', '2009-11.jsonl.zst', '2006-09.jsonl.zst', '2008-12.jsonl.zst', '2006-07.jsonl.zst', '2008-06.jsonl.zst', '2010-06.jsonl.zst', '2015-08.jsonl.zst', '2010-07.jsonl.zst', '2007-01.jsonl.zst', '2010-11.jsonl.zst', '2007-12.jsonl.zst', '2018-05.jsonl.zst', '2005-08.jsonl.zst', '2016-08.jsonl.zst', '2016-09.jsonl.zst', '2015-06.jsonl.zst', '2018-09.jsonl.zst', '2019-07.jsonl.zst', '2010-12.jsonl.zst', '2013-04.jsonl.zst', '2019-11.jsonl.zst', '2007-07.jsonl.zst', '2016-06.jsonl.zst', '2017-10.jsonl.zst', '2018-08.jsonl.zst', '2019-12.jsonl.zst', '2011-08.jsonl.zst', '2017-03.jsonl.zst', '2017-07.jsonl.zst', '2006-08.jsonl.zst', '2009-10.jsonl.zst', '2016-02.jsonl.zst', '2014-02.jsonl.zst', '2009-02.jsonl.zst', '2015-09.jsonl.zst', '2011-03.jsonl.zst', '2012-02.jsonl.zst', '2018-10.jsonl.zst', '2005-06.jsonl.zst', '2015-01.jsonl.zst', '2006-10.jsonl.zst', '2016-10.jsonl.zst', '2015-11.jsonl.zst', '2013-10.jsonl.zst', '2010-10.jsonl.zst', '2018-06.jsonl.zst', '2012-05.jso

In [8]:
# Process all files
if files_processed_to_text == False:
    with open(output_file, "w", encoding="utf-8") as outf:
        for filename in tqdm(files, total=len(files), desc="Processing Files"):
            print(f"{showTime()} Processing: {filename}")
            file_path = os.path.join(folder_path, filename)
            try:
                for text_line in decompress_zst_to_text(file_path, vocab, mode="accuracy"):
                    outf.write(text_line.strip())  # Write only the text line
            except Exception as e:
                print(f"Error processing {file_path}: {e}")

Processing Files:   0%|          | 0/179 [00:00<?, ?it/s]

[2025-04-30 15:04:29.475609 UTC] Processing: 2018-02.jsonl.zst


Processing Files:   1%|          | 1/179 [07:46<23:05:13, 466.93s/it]

[2025-04-30 15:12:16.404731 UTC] Processing: 2009-11.jsonl.zst


Processing Files:   1%|          | 2/179 [08:17<10:20:39, 210.39s/it]

...

Processing Files:  99%|█████████▉| 177/179 [11:02:46<01:59, 59.80s/it]

[2025-05-01 02:07:16.067231 UTC] Processing: 2007-10.jsonl.zst


Processing Files:  99%|█████████▉| 178/179 [11:02:55<00:44, 44.58s/it]

[2025-05-01 02:07:25.131982 UTC] Processing: 2019-01.jsonl.zst


Processing Files: 100%|██████████| 179/179 [11:12:30<00:00, 225.42s/it]


In [9]:
# Write vocabulary
with open(vocab_file, "w", encoding="utf-8") as vfile:
    for char in sorted(vocab):
        vfile.write(char + "\n")

In [ ]:
#load sequence
with open(output_file, "r", encoding="utf-8") as f:
    number_of_characters_to_read = 1_000_000
    text_sequence = f.read(number_of_characters_to_read)

len(text_sequence)

In [11]:
# Karpathy minBPE repository
from minbpe import RegexTokenizer

tokenizer = RegexTokenizer()
tokenizer.train(text_sequence, vocab_size=16_384)

In [ ]:
vocab = tokenizer.vocab
vocab

In [ ]:
tokenizer.encode("Hello, world! I like apple juice - I drink it every day. Isn't that too much?")

In [ ]:
tokenizer.decode([72,
 464,
 111,
 44,
 916,
 33,
 333,
 693,
 601,
 297,
 459,
 117,
 477,
 1007,
 333,
 743,
 857,
 343,
 833,
 289,
 331,
 46,
 333,
 115,
 110,
 785,
 318,
 286,
 111,
 961,
 63])

In [15]:
max_vocab_id = list(tokenizer.vocab.keys())[-1]
tokenizer.special_tokens = {
    "<|startoftext|>": max_vocab_id + 1,
    "<|separator|>": max_vocab_id + 2,
    "<|endoftext|>": max_vocab_id + 3,
    "<|unk|>": max_vocab_id + 4,
    "<|padding|>": max_vocab_id + 5
}

In [22]:
tokenizer_output_dir = "output_v6/tokenizer"
import os
if not os.path.exists(tokenizer_output_dir):
    os.makedirs(tokenizer_output_dir)

tokenizer_path = os.path.join(tokenizer_output_dir, "darija_tokenizer")
tokenizer.save(file_prefix=tokenizer_path)

In [ ]:
# Encoding the sequence of text

from minbpe import RegexTokenizer

tokenizer = RegexTokenizer()
tokenizer.load(model_file=tokenizer_path+".model")

In [ ]:
# Encode the data in batches

encoded_text_sequence = []
batch_size = 100_000_000
file_path = "output_val_v3.txt"

with open(file_path, "r", encoding="utf-8") as f:
    while True:
        chunk = f.read(batch_size)
        if not chunk:
            break
        batch_tokens = tokenizer.encode(chunk)
        encoded_text_sequence.extend(batch_tokens)
        print(f"{showTime()} Processed {len(encoded_text_sequence)} tokens so far.")

print(f"Total tokens: {len(encoded_text_sequence)}")

In [29]:
import numpy as np


encoder_output_dir = "output_v6/encoded_data"
import os
if not os.path.exists(encoder_output_dir):
    os.makedirs(encoder_output_dir)

output_path = os.path.join(encoder_output_dir, "encoded_output_val_v3.npy")
np.save(output_path, np.array(encoded_text_sequence, dtype=np.int64))

# Free up memory
del encoded_text_sequence